In [1]:
import torch

from torch_openreml import MixedModelREML
from torch_openreml.covariance import DummyMatrix, ScalarMatrix

n, p = 10, 2

group = ["a", "b"] * 5
effect = torch.tensor([1.0, -1.0]).repeat(n // 2)

X = torch.randn(n, p)
y = X @ torch.tensor([0.5, -0.5]) + effect + 0.5 * torch.randn(n)

reml = MixedModelREML(
    genotype=(DummyMatrix(group), ScalarMatrix(group)),
)

names, free_params = reml.optimize(y, X, verbose=1)
names, free_params

[  1] ∥∇∥:       5.5900, ∥Δ∥: 6.5348, η: 0.25, ∥Δᶜ∥: 1.6337, accepted: True , log 𝓛:  -0.4265
[  2] ∥∇∥:       4.5979, ∥Δ∥: 0.4375, η: 1.00, ∥Δᶜ∥: 0.4375, accepted: True , log 𝓛:   0.2352
[  3] ∥∇∥:       0.9439, ∥Δ∥: 0.1063, η: 1.00, ∥Δᶜ∥: 0.1063, accepted: True , log 𝓛:   0.2812
[  4] ∥∇∥:       0.0605, ∥Δ∥: 0.0093, η: 1.00, ∥Δᶜ∥: 0.0093, accepted: True , log 𝓛:   0.2814
[  5] ∥∇∥:       0.0003, ∥Δ∥: 0.0001, η: 0.00, ∥Δᶜ∥: 0.0000, accepted: True , log 𝓛:   0.2814

[∇: score, Δ: 𝐉⁻¹∇, η: step length, Δᶜ: clip(𝛉 + ηΔ, lb, ub) - 𝛉, 𝓛: restricted likelihood]

✓ Converged at iteration 5, total time: 0.01s



(['G/sigma^2', 'R/sigma^2'], tensor([ 0.1023, -1.3062]))

In [2]:
history = reml.history
history["loglik"]

[tensor(-0.4265),
 tensor(0.2352),
 tensor(0.2812),
 tensor(0.2814),
 tensor(0.2814)]

In [3]:
torch.stack(history["loglik"])

tensor([-0.4265,  0.2352,  0.2812,  0.2814,  0.2814])

In [4]:
torch.stack(history["free_params"])

tensor([[ 0.3885, -1.5868],
        [ 0.0073, -1.3720],
        [ 0.0939, -1.3104],
        [ 0.1023, -1.3062],
        [ 0.1023, -1.3062]])

In [5]:
names, theta_hat = reml.get_free_params("last")
names, theta_hat

(['G/sigma^2', 'R/sigma^2'], tensor([ 0.1023, -1.3062]))

In [6]:
params = reml.build_params("last", out_format="dict")
params

{'G/sigma^2': tensor([1.2270]), 'R/sigma^2': tensor([0.0734])}

In [7]:
tree = reml.tree("last")
tree

{'/': tensor([[1.3004, 0.0000, 1.2270, 0.0000, 1.2270, 0.0000, 1.2270, 0.0000, 1.2270,
          0.0000],
         [0.0000, 1.3004, 0.0000, 1.2270, 0.0000, 1.2270, 0.0000, 1.2270, 0.0000,
          1.2270],
         [1.2270, 0.0000, 1.3004, 0.0000, 1.2270, 0.0000, 1.2270, 0.0000, 1.2270,
          0.0000],
         [0.0000, 1.2270, 0.0000, 1.3004, 0.0000, 1.2270, 0.0000, 1.2270, 0.0000,
          1.2270],
         [1.2270, 0.0000, 1.2270, 0.0000, 1.3004, 0.0000, 1.2270, 0.0000, 1.2270,
          0.0000],
         [0.0000, 1.2270, 0.0000, 1.2270, 0.0000, 1.3004, 0.0000, 1.2270, 0.0000,
          1.2270],
         [1.2270, 0.0000, 1.2270, 0.0000, 1.2270, 0.0000, 1.3004, 0.0000, 1.2270,
          0.0000],
         [0.0000, 1.2270, 0.0000, 1.2270, 0.0000, 1.2270, 0.0000, 1.3004, 0.0000,
          1.2270],
         [1.2270, 0.0000, 1.2270, 0.0000, 1.2270, 0.0000, 1.2270, 0.0000, 1.3004,
          0.0000],
         [0.0000, 1.2270, 0.0000, 1.2270, 0.0000, 1.2270, 0.0000, 1.2270, 0.0000,
    

In [8]:
beta_hat = reml.blue(y, X, theta_hat)
beta_hat

tensor([ 0.5144, -0.4646])

In [9]:
cov_beta = reml.cov_blue(X, theta_hat)
cov_beta

tensor([[ 0.0123, -0.0010],
        [-0.0010,  0.0139]])

In [10]:
se_beta = torch.sqrt(reml.var_blue(X, theta_hat))
se_beta

tensor([0.1107, 0.1178])

In [11]:
b_hat = reml.blup()
b_hat

tensor([ 1.0829, -1.1149])

In [12]:
b_cov = reml.cov_blup()
b_cov

tensor([[ 1.2092, -0.0017],
        [-0.0017,  1.2062]])

In [13]:
b_var = reml.var_blup()
b_var

tensor([1.2092, 1.2062])

In [14]:
b_by_term = reml.blup_per_term()
b_by_term

{'genotype': tensor([ 1.0829, -1.1149])}

In [15]:
b_cov_by_term = reml.cov_blup_per_term()
b_cov_by_term

{'genotype': tensor([[ 1.2092, -0.0017],
         [-0.0017,  1.2062]])}

In [16]:
b_var_by_term = reml.var_blup_per_term()
b_var_by_term

{'genotype': tensor([1.2092, 1.2062])}

In [17]:
y_marginal = reml.marginal_predict()
y_marginal

tensor([ 0.0835,  0.0092,  0.0872,  0.4813, -0.6429,  0.2009,  0.7887,  0.1797,
         1.3188,  0.6532])

In [18]:
y_conditional = reml.predict()
y_conditional

tensor([ 1.1664, -1.1056,  1.1701, -0.6336,  0.4401, -0.9140,  1.8716, -0.9351,
         2.4017, -0.4616])

In [19]:
y_new = reml.predict_new(
    X[:2],
    {"genotype": torch.tensor([[1.0, 0.0], [0.0, 1.0]])},
)
y_new

tensor([ 1.1664, -1.1056])

In [20]:
z_new = DummyMatrix(group).transform(["a", "b"])

y_new = reml.predict_new(
    X[:2],
    z_new,
)
y_new

tensor([ 1.1664, -1.1056])

In [21]:
e_marginal = reml.marginal_residual(y, X, theta_hat)
e_marginal

tensor([ 0.5937, -1.0803,  1.4503, -0.9564,  1.2003, -1.2253,  1.1117, -1.2232,
         1.1234, -1.1557])

In [22]:
e_conditional = reml.residual(y, X, theta_hat)
e_conditional

tensor([-0.4892,  0.0345,  0.3674,  0.1584,  0.1174, -0.1104,  0.0288, -0.1083,
         0.0405, -0.0408])

In [23]:
c11, c12, c21, c22 = reml.c_inverse_blocks(X, theta_hat)
c11, c12, c21, c22

(tensor([[ 0.0123, -0.0010],
         [-0.0010,  0.0139]]),
 tensor([[-0.0060, -0.0004],
         [ 0.0028,  0.0094]]),
 tensor([[-0.0060,  0.0028],
         [-0.0004,  0.0094]]),
 tensor([[0.0178, 0.0017],
         [0.0017, 0.0208]]))

In [24]:
loglik_value, loglik_method = reml.loglik(y, X, theta_hat)
loglik_value, loglik_method

(tensor(0.2814), 'cform')

In [25]:
reml.is_converged()

True